# Workflow Planning Evaluator — Bug Bash Notebook

This notebook walks you through:
1. Installing required packages
2. Setting up tracing with Application Insights
3. Running 3 multi-agent HR recruitment workflows:
   - **Group Chat** with agent manager (orchestrator selects speakers)
   - **Magentic** multi-source sourcing with compliance
   - **Sequential + Human-in-the-Loop** hiring pipeline
4. Fetching traces from App Insights and converting them
5. Running the Workflow Planning Evaluator on each trace

**Prerequisites:**
- Azure AI project with Application Insights attached
- Run `az login` before starting
- Set your environment variables in the cell below

## 1. Install Packages

`pip install agent-framework==1.0.0b251211 agent-framework-azure-ai==1.0.0rc3 azure-ai-projects==2.0.0b4`

`pip install -e ../../../../../../..[redteam]`

`pip install azure-monitor-opentelemetry azure-monitor-query azure-identity opentelemetry-api python-dotenv`

## 2. Environment Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# ── Set these environment variables (or use a .env file) ──
# os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"] = ""
# os.environ["APP_INSIGHTS_WORKSPACE_ID"] = ""
# os.environ["AZURE_AI_PROJECT_ENDPOINT"] = ""
# os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"] = ""

# For evaluation
# os.environ["EVAL_GPT4_1_ENDPOINT"] = ""
# os.environ["EVAL_GPT4_1_API_KEY"] = ""
# os.environ["EVAL_GPT4_1_API_VERSION"] = ""
# os.environ["EVAL_GPT4_1_DEPLOYMENT_NAME"] = ""

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


## 3. Tracing Setup

Configure Azure Monitor telemetry and enable instrumentation with sensitive data capture.

In [2]:
from agent_framework.observability import enable_instrumentation, get_tracer
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

# Configure Azure Monitor exporter (uses APPLICATIONINSIGHTS_CONNECTION_STRING from .env)
configure_azure_monitor(
    connection_string=os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING"),
    enable_live_metrics=True,
)
enable_instrumentation(enable_sensitive_data=True)

tracer = get_tracer()
print("Tracing configured with App Insights + sensitive data enabled!")

Tracing configured with App Insights + sensitive data enabled!


In [3]:
# We'll collect trace IDs from each workflow run
collected_trace_ids = {}

## 4. Import HR Agents & Tools

The HR mock data, function tools, and agent factories are defined in `scripts/hr_tools.py` and `scripts/hr_agents.py`.

In [4]:
from scripts.hr_agents import (
    cleanup_clients,
    create_compliance_guard,
    create_evaluator,
    create_magentic_manager,
    create_mobility_scout,
    create_orchestrator,
    create_req_master,
    create_scheduler,
    create_talent_scout,
)

print("HR agents and tools loaded!")

HR agents and tools loaded!


## 5. Workflow 1: Group Chat with Agent Manager

An HR group chat where an **Orchestrator agent** decides which specialist speaks next.
The team works together to find candidates for a Senior Software Engineer position.

**Feel free to modify the agents, task, and termination condition!**

In [5]:
from typing import cast
from agent_framework import AgentResponseUpdate, Message, WorkflowEvent
from agent_framework.orchestrations import GroupChatBuilder

# ── Create HR agents (each gets its own AzureAIClient) ──
orchestrator = await create_orchestrator()
req_master = await create_req_master()
talent_scout = await create_talent_scout()
evaluator = await create_evaluator()

# ── Run with tracing (build + run inside the same span so workflow.build is captured) ──
task = (
    "Help me fill the Senior Software Engineer position JOB-SWE-2025-001. "
    "Find candidates, evaluate them, and recommend a shortlist."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("group_chat_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["group_chat"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    # Build the group chat workflow
    group_chat_workflow = (
        GroupChatBuilder(
            participants=[req_master, talent_scout, evaluator],
            max_rounds=20,
            intermediate_outputs=True,
            orchestrator_agent=orchestrator,
        )
        .build()
    )

    last_response_id = None
    async for event in group_chat_workflow.run(task, stream=True):
        if event.type == "output" and not isinstance(event.data, AgentResponseUpdate):
            outputs = cast(list[Message], event.data)
            print("\n" + "=" * 80)
            print("\nFinal Conversation:")
            for message in outputs:
                print(f"  [AGENT {message.author_name or message.role}]: {message.text[:200]}...\n")

print(f"\n✅ Group Chat complete! Trace ID: {collected_trace_ids['group_chat']}")


Task: Help me fill the Senior Software Engineer position JOB-SWE-2025-001. Find candidates, evaluate them, and recommend a shortlist.
Trace ID: 8754b9914800415eb7b9993755061fc6



Final Conversation:
  [AGENT user]: Help me fill the Senior Software Engineer position JOB-SWE-2025-001. Find candidates, evaluate them, and recommend a shortlist....

  [AGENT ReqMaster]: Senior Software Engineer – JOB-SWE-2025-001 (Validated and Approved)
Status: Open for candidate sourcing
Urgency: High

**Job Overview:**
- Department: Engineering
- Location: Seattle, WA (Remote allo...

  [AGENT TalentScout]: **Candidate Screening and Evaluation: Senior Software Engineer | JOB-SWE-2025-001**

---

### 1. Alice Zhang – CAND-EXT-1001
- **Location:** Seattle, WA
- **Experience:** 7 years
- **Skills:** Python,...

  [AGENT Evaluator]: ## Ranked Shortlist: Senior Software Engineer (JOB-SWE-2025-001)

### 1. Bob Patel (CAND-EXT-1002) — Score: 59/100
- **Location:** San Francisco, CA (open to remote)
- **Years 

## 6. Workflow 2: Magentic Multi-Source Sourcing

A Magentic orchestration where a manager coordinates multiple specialists to source
candidates from all channels, evaluate them, and run compliance checks.

**Feel free to modify the agents, task, and configuration!**

In [6]:
import json as json_module
from agent_framework.orchestrations import GroupChatRequestSentEvent, MagenticBuilder, MagenticProgressLedger

# ── Create HR agents (each gets its own AzureAIClient) ──
magentic_manager = await create_magentic_manager()
mag_talent_scout = await create_talent_scout()
mag_mobility_scout = await create_mobility_scout()
mag_evaluator = await create_evaluator()
mag_compliance = await create_compliance_guard()

# ── Run with tracing (build + run inside the same span so workflow.build is captured) ──
task = (
    "We need to hire for JOB-SWE-2025-001 (Senior Software Engineer). "
    "Source candidates from all channels (external job boards and internal transfers), "
    "evaluate them, and run a compliance check on the final shortlist."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("magentic_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["magentic"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    # Build the Magentic workflow
    magentic_workflow = MagenticBuilder(
        participants=[mag_talent_scout, mag_mobility_scout, mag_evaluator, mag_compliance],
        intermediate_outputs=True,
        manager_agent=magentic_manager,
        max_round_count=15,
    ).build()

    last_response_id = None
    output_event = None
    async for event in magentic_workflow.run(task, stream=True):
        if event.type == "output" and not isinstance(event.data, AgentResponseUpdate) and not event.type == "magentic_orchestrator":
            output_event = event

    if output_event:
        outputs = cast(list[Message], output_event.data)
        print("\n" + "=" * 80)
        print("\nFinal Conversation:")
        for message in outputs:
            print(f"  {message.author_name or message.role}: {message.text[:200]}...\n")

print(f"\n✅ Magentic complete! Trace ID: {collected_trace_ids['magentic']}")


Task: We need to hire for JOB-SWE-2025-001 (Senior Software Engineer). Source candidates from all channels (external job boards and internal transfers), evaluate them, and run a compliance check on the final shortlist.
Trace ID: b04125090e030820ef3fc0d57710a29f



Final Conversation:
  HiringManager: Here is the final outcome for your request to source, evaluate, and shortlist candidates for JOB-SWE-2025-001 (Senior Software Engineer):

---

### 1. Sourcing Summary
- **External candidates** were g...


✅ Magentic complete! Trace ID: b04125090e030820ef3fc0d57710a29f


## 7. Workflow 3: Sequential HR Pipeline with Human-in-the-Loop

A sequential hiring pipeline: ReqMaster \u2192 TalentScout \u2192 Evaluator \u2192 Scheduler.
The workflow **pauses after the Evaluator** so you can review the shortlist before
scheduling interviews.

**When prompted, provide feedback or type 'skip' to approve the shortlist.**

In [7]:
from collections.abc import AsyncIterable
from agent_framework import AgentExecutorResponse
from agent_framework.orchestrations import AgentRequestInfoResponse, SequentialBuilder


async def process_event_stream(stream: AsyncIterable[WorkflowEvent]):
    """Process events from the workflow stream, handling HITL requests."""
    requests = {}
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, AgentExecutorResponse):
            requests[event.request_id] = event.data
        elif event.type == "output":
            print("\n" + "=" * 60)
            print("WORKFLOW COMPLETE")
            print("=" * 60)
            outputs = cast(list[Message], event.data)
            for message in outputs:
                print(f"  [{message.author_name or message.role}]: {message.text[:300]}...")

    responses = {}
    if requests:
        for request_id, request in requests.items():
            print("\n" + "-" * 40)
            print(f"HUMAN REVIEW REQUESTED")
            print(f"Agent '{request.executor_id}' produced a shortlist:")
            print(f"  {request.agent_response.text[:500]}...")
            print("-" * 40)

            user_input = input("Your feedback (or 'skip' to approve): ")
            if user_input.lower() == "skip":
                responses[request_id] = AgentRequestInfoResponse.approve()
            else:
                responses[request_id] = AgentRequestInfoResponse.from_strings([user_input])

    return responses if responses else None


# ── Create HR agents (each gets its own AzureAIClient) ──
seq_req_master = await create_req_master()
seq_talent_scout = await create_talent_scout()
seq_evaluator = await create_evaluator()
seq_scheduler = await create_scheduler()

# ── Run with tracing (build + run inside the same span so workflow.build is captured) ──
task = (
    "Process the hiring pipeline for JOB-SWE-2025-001: gather requirements, "
    "source candidates, evaluate them (I'll review the shortlist), "
    "then schedule interviews for approved candidates."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("sequential_hitl_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["sequential_hitl"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    # Build the sequential workflow with HITL
    sequential_workflow = (
        SequentialBuilder(participants=[seq_req_master, seq_talent_scout, seq_evaluator, seq_scheduler])
        .with_request_info(agents=["Evaluator"])  # Pause after evaluator for human review
        .build()
    )

    stream = sequential_workflow.run(task, stream=True)
    pending_responses = await process_event_stream(stream)
    while pending_responses is not None:
        stream = sequential_workflow.run(stream=True, responses=pending_responses)
        pending_responses = await process_event_stream(stream)

print(f"\n✅ Sequential HITL complete! Trace ID: {collected_trace_ids['sequential_hitl']}")


Task: Process the hiring pipeline for JOB-SWE-2025-001: gather requirements, source candidates, evaluate them (I'll review the shortlist), then schedule interviews for approved candidates.
Trace ID: 28e66a82a8402d5b46f947a8e507e757


----------------------------------------
HUMAN REVIEW REQUESTED
Agent 'Evaluator' produced a shortlist:
  **Ranked Shortlist for Senior Software Engineer (JOB-SWE-2025-001)**

---

### 1. Bob Patel (CAND-EXT-1002)  
**Score:** 59/100  
**Location:** San Francisco, CA | **Remote:** Yes  
**Experience:** 9 years  
**Salary Expectation:** $195,000 (within budget)  
**Must-have Skills:** Python, Cloud (Azure/AWS)  
**Skill Gaps:** Distributed Systems not explicit; indirect evidence from backend/platform work  
**Nice-to-have:** Go, System Design  
**Evaluation Notes:**  
- Surpasses experience requireme...
----------------------------------------

WORKFLOW COMPLETE
  [user]: Process the hiring pipeline for JOB-SWE-2025-001: gather requirements, source candida

## 8. Summary of Collected Trace IDs

In [8]:
print("Collected Trace IDs:")
for name, tid in collected_trace_ids.items():
    print(f"  {name}: {tid}")

all_trace_ids = list(collected_trace_ids.values())
print(f"\nTotal: {len(all_trace_ids)} traces")

Collected Trace IDs:
  group_chat: 8754b9914800415eb7b9993755061fc6
  magentic: b04125090e030820ef3fc0d57710a29f
  sequential_hitl: 28e66a82a8402d5b46f947a8e507e757

Total: 3 traces


## 9. Wait for App Insights Ingestion

Application Insights has ~90 second ingestion latency. We wait before querying.

In [9]:
import time

WAIT_SECONDS = 90
print(f"Waiting {WAIT_SECONDS} seconds for App Insights ingestion...")
for i in range(WAIT_SECONDS, 0, -10):
    print(f"  {i}s remaining...", flush=True)
    time.sleep(min(10, i))
print("Done waiting!")

Waiting 90 seconds for App Insights ingestion...
  90s remaining...
  80s remaining...
  70s remaining...
  60s remaining...
  50s remaining...
  40s remaining...
  30s remaining...
  20s remaining...
  10s remaining...
Done waiting!


## 10. Fetch & Convert Traces

Query App Insights for each trace ID, then convert to the structured format expected by the evaluator.

In [10]:
from scripts.trace_query import query_traces, process_workflow_trace_rows
from scripts.workflow_trace_converter import convert_workflow_traces

workspace_id = os.environ["APP_INSIGHTS_WORKSPACE_ID"]
converted_traces = {}

for workflow_name, trace_id in collected_trace_ids.items():
    print(f"\n{'=' * 60}")
    print(f"Fetching traces for: {workflow_name} (trace_id: {trace_id})")
    print(f"{'=' * 60}")

    try:
        raw_rows = query_traces(
            workspace_id=workspace_id,
            trace_ids=[trace_id],
            lookback_hours=1,
        )
        print(f"  Retrieved {len(raw_rows)} raw rows")

        spans = process_workflow_trace_rows(raw_rows)
        print(f"  Processed into {len(spans)} spans")

        converted = convert_workflow_traces(spans)
        converted_traces[workflow_name] = converted

        print(f"  Workflow type: {converted.get('workflow_type', 'N/A')}")
        print(f"  Agents: {list(converted.get('agents', {}).keys())}")
        print(f"  Invocations: {len(converted.get('invocations', []))}")
        print(f"  Parse failed: {converted.get('parse_failed', False)}")
        print(f"  \u2705 Conversion successful!")
    except Exception as e:
        print(f"  \u274c Error: {e}")
        converted_traces[workflow_name] = None

print(f"\n\nSuccessfully converted: {sum(1 for v in converted_traces.values() if v is not None)}/{len(converted_traces)}")


Fetching traces for: group_chat (trace_id: 8754b9914800415eb7b9993755061fc6)
  Retrieved 113 raw rows
  Processed into 113 spans
  Workflow type: group_chat
  Agents: ['Orchestrator', 'ReqMaster', 'TalentScout', 'Evaluator']
  Invocations: 7
  Parse failed: False
  ✅ Conversion successful!

Fetching traces for: magentic (trace_id: b04125090e030820ef3fc0d57710a29f)
  Retrieved 185 raw rows
  Processed into 185 spans
  Workflow type: magentic
  Agents: ['HiringManager', 'TalentScout', 'MobilityScout', 'Evaluator', 'ComplianceGuard']
  Invocations: 12
  Parse failed: False
  ✅ Conversion successful!

Fetching traces for: sequential_hitl (trace_id: 28e66a82a8402d5b46f947a8e507e757)
  Retrieved 73 raw rows
  Processed into 73 spans
  Workflow type: sequential
  Agents: ['ReqMaster', 'TalentScout', 'Evaluator', 'Scheduler']
  Invocations: 4
  Parse failed: False
  ✅ Conversion successful!


Successfully converted: 3/3


## 11. (Optional) Inspect Converted and Reformatted Traces

View the converted trace structure for debugging.

First cells shows the converted traces.

Second cell shows the reformatted traces (direct input to the evaluator).

In [11]:
import json

for workflow_name, trace_data in converted_traces.items():
    if trace_data is None:
        continue
    print(f"\n{'=' * 60}")
    print(f"{workflow_name} \u2014 Converted Trace")
    print(f"{'=' * 60}")
    print(json.dumps(trace_data, indent=2, default=str)[:3000])
    print("..." * 60)


group_chat — Converted Trace
{
  "parse_failed": false,
  "workflow_id": "4c548693-dd8e-4a75-9deb-f7d4951dbba3",
  "workflow_name": "WorkflowBuilder-42491a18-eb0e-4db4-872c-4225b72e153f",
  "workflow_type": "group_chat",
  "topology": {
    "start_executor_id": "Orchestrator",
    "executors": [
      {
        "id": "Orchestrator",
        "type": "AgentBasedGroupChatOrchestrator"
      },
      {
        "id": "ReqMaster",
        "type": "AgentExecutor"
      },
      {
        "id": "TalentScout",
        "type": "AgentExecutor"
      },
      {
        "id": "Evaluator",
        "type": "AgentExecutor"
      }
    ],
    "edges": [
      {
        "type": "SingleEdgeGroup",
        "source": "Orchestrator",
        "target": "ReqMaster"
      },
      {
        "type": "SingleEdgeGroup",
        "source": "ReqMaster",
        "target": "Orchestrator"
      },
      {
        "type": "SingleEdgeGroup",
        "source": "Orchestrator",
        "target": "TalentScout"
      },
    

In [12]:
from azure.ai.evaluation._evaluators._workflow_planning._utils import format_workflow_trace_for_eval

# Show evaluator-side reformatted trace text
workflow_name = "group_chat"
# workflow_name = "magentic"
# workflow_name = "sequential_hitl"

trace_data = converted_traces.get(workflow_name)



if trace_data is None:
    print(f"No converted trace found for '{workflow_name}'.")
else:
    reformatted = format_workflow_trace_for_eval(trace_data)
    print(f"Reformatted trace preview for '{workflow_name}':")
    print("=" * 80)
    print(reformatted)

Reformatted trace preview for 'group_chat':
[Workflow Definition]
Executors: Orchestrator (AgentBasedGroupChatOrchestrator), ReqMaster (AgentExecutor), TalentScout (AgentExecutor), Evaluator (AgentExecutor)
Edges:
  Orchestrator -> ReqMaster
  ReqMaster -> Orchestrator
  Orchestrator -> TalentScout
  TalentScout -> Orchestrator
  Orchestrator -> Evaluator
  Evaluator -> Orchestrator

----------------------------------------

[User Input]
Help me fill the Senior Software Engineer position JOB-SWE-2025-001. Find candidates, evaluate them, and recommend a shortlist.

----------------------------------------

[Orchestrator - Invocation 1]
  [System Prompt]
  You coordinate an HR recruitment team conversation to fulfill hiring requests.

Guidelines:
- Start with ReqMaster to gather job requirements
- Then have TalentScout source external candidates
- Use Evaluator to score and rank candidates
- Only finish after a shortlist has been produced with clear recommendations
- Keep the conversatio

## 12. Run Workflow Planning Evaluator

Run the evaluator on each converted trace and display results.

In [13]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.ai.evaluation._evaluators._workflow_planning import _WorkflowPlanningEvaluator

# Configure the model for the evaluator
model_config = AzureOpenAIModelConfiguration(
            azure_endpoint=os.environ["EVAL_GPT4_1_ENDPOINT"],
            api_key=os.environ["EVAL_GPT4_1_API_KEY"],
            api_version=os.environ["EVAL_GPT4_1_API_VERSION"],
            azure_deployment=os.environ["EVAL_GPT4_1_DEPLOYMENT_NAME"],
)

evaluator = _WorkflowPlanningEvaluator(model_config=model_config)

results = {}
for workflow_name, trace_data in converted_traces.items():
    if trace_data is None:
        print(f"\n\u23ed\ufe0f  Skipping {workflow_name} (no trace data)")
        continue

    print(f"\n{'=' * 60}")
    print(f"Evaluating: {workflow_name}")
    print(f"{'=' * 60}")

    try:
        result = evaluator(workflow_trace=trace_data)
        results[workflow_name] = result

        print(f"  Score:  {result.get('workflow_planning', 'N/A')}")
        print(f"  Result: {result.get('workflow_planning_result', 'N/A')}")
        print(f"  Reason: {result.get('workflow_planning_reason', 'N/A')}")
        print(f"  \u2705 Evaluation complete!")
    except Exception as e:
        print(f"  \u274c Error: {e}")
        results[workflow_name] = {"error": str(e)}


Evaluating: group_chat
  Score:  1
  Result: pass
  Reason: The workflow demonstrated strong planning: each hiring sub-task was routed to the agent best equipped for that stage, outputs were clearly handed off, progress was sequentially tracked based on prior results, and no critical errors or loops occurred.
  ✅ Evaluation complete!

Evaluating: magentic
  Score:  1
  Result: pass
  Reason: This workflow was well-planned: each objective was decomposed into sourcing, evaluation, and compliance sub-tasks, with agents assigned logically. Agent handoffs reflected prior outputs, with no loops or stagnation. Progress and compliance were tracked, and error handling was adequate throughout.
  ✅ Evaluation complete!

Evaluating: sequential_hitl
  Score:  1
  Result: pass
  Reason: The workflow was well-planned, following a clear, fixed sequential structure: requirements gathering, sourcing, evaluation, and scheduling. Each agent was correctly assigned, stages occurred in logical order, depend

## Cleanup

In [14]:
await cleanup_clients()
print("Cleaned up all agent clients!")

Cleaned up all agent clients!
